# Generate a nextnano Input File from a PHIDL Double Quantum Dot Layout

This notebook demonstrates the full translation pipeline for a **double quantum dot device**:

- build a parametric 2D gate layout using PHIDL-based device builders
- export the GDS/SVG layout
- extract the structured layout specification
- define the vertical process stack
- build a 3D simulation layout
- write a nextnano++ `.in` file from a reference template
- optionally run the simulation with `nextnanopy`
- inspect the generated structure and output folders

## Setup

Import the reusable project APIs and locate the repository independently of the Jupyter launch directory.

In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from phidl import quickplot as qp

from qd_design import (
    LinearDotArrayDevice,
    build_simulation_layout,
    make_reference_sige_ge_process_stack,
    write_nextnano_input_from_template,
)
from nextnanopp_tools import (
    convergence_summary,
    find_probability_peaks,
    find_vtr_plane_extrema,
    get_bias_dir,
    get_output_directory,
    integrated_density_region_columns,
    list_variables,
    load_vtr_linecut,
    load_vtr_plane,
    plot_bias_volume_3d,
    plot_bias_volume_linecut,
    plot_bias_volume_slice,
    plot_convergence,
    plot_integrated_density_hole,
    plot_quantum_density_volume_3d,
    plot_quantum_density_volume_linecut,
    plot_quantum_density_volume_slice,
    plot_quantum_energy_spectrum,
    plot_quantum_occupation,
    plot_quantum_probability_volume_3d,
    plot_quantum_probability_volume_linecut,
    plot_quantum_probability_volume_slice,
    plot_structure_plane,
    plot_total_charges,
    read_index_table,
    read_integrated_density_hole,
    read_quantum_energy_spectrum,
    read_quantum_occupation,
    read_total_charges,
    resolve_bias_output_file,
    resolve_quantum_output_file,
    resolve_quantum_probability_state_file,
    resolve_structure_file,
    run_input_file,
    validate_run_directory,
)

START_PATH = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (START_PATH, *START_PATH.parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(
        f"Could not locate the repository root from {START_PATH}; "
        "expected a parent containing pyproject.toml and src/."
    )

### Internal generated-file paths

In [ ]:
# Repository paths are anchored independently of the Jupyter launch directory.
OUTPUT_DIR = REPO_ROOT / "data" / "gds"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENERATED_INPUT_DIR = REPO_ROOT / "configs" / "robert_inputs" / "generated"
GENERATED_INPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reference nextnano input file used as template.
# Adjust this path if your template lives elsewhere.
TEMPLATE_INPUT_PATH = (
    REPO_ROOT
    / "configs"
    / "robert_inputs"
    / "double_qd"
    / "3d"
    / "Double_Quantum_Dot_3D.in"
)

# Generated files.
GDS_PATH = OUTPUT_DIR / "double_dot_from_phidl.gds"
SVG_PATH = OUTPUT_DIR / "double_dot_from_phidl.svg"
LAYOUT_SPEC_PATH = OUTPUT_DIR / "double_dot_from_phidl_layout_spec.json"
SIMULATION_LAYOUT_PATH = OUTPUT_DIR / "double_dot_from_phidl_simulation_layout.json"
GENERATED_INPUT_PATH = GENERATED_INPUT_DIR / "Double_Quantum_Dot_3D_from_PHIDL.in"

print("Template exists:", TEMPLATE_INPUT_PATH.exists())
print("Template path:", TEMPLATE_INPUT_PATH.resolve())
print("Generated input path:", GENERATED_INPUT_PATH.resolve())

## User controls

For local execution, set `RUN_SIMULATION=True` and `RUN_DIRECTORY=None`. Otherwise keep execution disabled and replace the placeholder with one explicit completed run directory.

In [ ]:
RUN_SIMULATION = False
RUN_DIRECTORY = Path("/Users/robertjovanov/code/qpu-design-automation-toolkit/runs/Double_Quantum_Dot_3D__20260429_004136")
BIAS = "bias_00000"
ANALYSE_QUANTUM_OUTPUTS = True
EXPORT_FIGURES = False

## Device layout

Build the double-dot geometry from the existing parametric device definition.

In [ ]:
double_dot = LinearDotArrayDevice(
    name="double_dot_from_phidl",
    n_dots=2,

    device_y_size_nm=200.0,

    ohmic_width_nm=40.0,
    ohmic_length_nm=200.0,

    barrier_width_nm=40.0,
    barrier_length_nm=140.0,

    plunger_body_width_nm=40.0,
    plunger_body_length_nm=50.0,
    plunger_head_top_width_nm=60.0,
    plunger_head_max_width_nm=100.0,
    plunger_head_height_nm=100.0,
    plunger_upper_taper_height_nm=25.0,
    plunger_lower_taper_height_nm=25.0,

    ohmic_to_barrier_gap_nm=20.0,
    barrier_to_plunger_gap_nm=20.0,
)

layout = double_dot.ensure_built()

print("\n" + "=" * 80)
print("Double Dot Device Summary")
print("=" * 80)
print(json.dumps(double_dot.summary(), indent=2))

### Intended PHIDL geometry

In [ ]:
qp(layout)

### Export GDS and SVG

In [ ]:
double_dot.write_gds(str(GDS_PATH))
double_dot.write_svg(str(SVG_PATH))

print(f"Saved GDS: {GDS_PATH.resolve()}")
print(f"Saved SVG: {SVG_PATH.resolve()}")

### Structured layout specification

Inspect and save the geometry representation used to build the simulation layout.

In [ ]:
layout_spec = double_dot.layout_spec()

print("\n" + "=" * 80)
print("Layout Spec")
print("=" * 80)
for element in layout_spec:
    print(
        element["name"],
        element["gate_type"],
        element["layer_name"],
        f"x=[{element['x_min_nm']}, {element['x_max_nm']}]",
        f"y=[{element['y_min_nm']}, {element['y_max_nm']}]",
        f"num_polygons={len(element.get('polygon_xy_nm', []))}",
    )

#### Save the layout specification

In [ ]:
with open(LAYOUT_SPEC_PATH, "w", encoding="utf-8") as f:
    json.dump(layout_spec, f, indent=2)

print(f"Saved layout spec: {LAYOUT_SPEC_PATH.resolve()}")

## Process stack and simulation layout

Create the established SiGe/Ge process stack, then combine it with the layout specification.

In [ ]:
process_stack = make_reference_sige_ge_process_stack()

print("\n" + "=" * 80)
print("Process Stack")
print("=" * 80)
print(json.dumps(process_stack.to_dict(), indent=2))

### Build the 3D simulation layout

In [ ]:
simulation_layout = build_simulation_layout(
    name="double_dot_simulation_layout",
    layout_elements=layout_spec,
    process_stack=process_stack,
    x_margin_nm=0.0,
    y_margin_nm=0.0,
)

print("\n" + "=" * 80)
print("Simulation Layout Domain")
print("=" * 80)
print(json.dumps(simulation_layout.domain.to_dict(), indent=2))

### Inspect background and patterned regions

In [ ]:
print("\n" + "=" * 80)
print("Background Regions")
print("=" * 80)
for region in simulation_layout.background_regions:
    print(
        region.name,
        region.material,
        f"z=[{region.z_min_nm}, {region.z_max_nm}]",
        f"x=[{region.x_min_nm}, {region.x_max_nm}]",
        f"y=[{region.y_min_nm}, {region.y_max_nm}]",
    )

print("\n" + "=" * 80)
print("Patterned Regions")
print("=" * 80)
for region in simulation_layout.patterned_regions:
    print(
        region.name,
        region.gate_type,
        region.layer_name,
        region.material,
        f"x=[{region.x_min_nm}, {region.x_max_nm}]",
        f"y=[{region.y_min_nm}, {region.y_max_nm}]",
        f"z=[{region.z_min_nm}, {region.z_max_nm}]",
        f"num_polygons={len(region.polygon_xy_nm)}",
    )

### Save the simulation layout

In [ ]:
simulation_layout.write_json(str(SIMULATION_LAYOUT_PATH))

print(f"Saved simulation layout: {SIMULATION_LAYOUT_PATH.resolve()}")

## Input generation

Generate the nextnano++ input from the existing template and unchanged voltage overrides.

In [ ]:
if not TEMPLATE_INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Template input file not found:\n{TEMPLATE_INPUT_PATH.resolve()}\n\n"
        "Copy your reference Double_Quantum_Dot_3D.in file to this path, "
        "or update TEMPLATE_INPUT_PATH in the config cell."
    )

write_nextnano_input_from_template(
    simulation_layout=simulation_layout,
    template_path=TEMPLATE_INPUT_PATH,
    output_path=GENERATED_INPUT_PATH,
    voltage_overrides={
        "V_P1": -3.0,
        "V_P2": -3.0,
        "V_B1": 0.0,
        "V_B2": 0.0,
        "V_B3": 0.0,
        "V_OC_L": 0.0,
        "V_OC_R": 0.0,
    },
)

print(f"Generated nextnano input file: {GENERATED_INPUT_PATH.resolve()}")

### Inspect the generated input snippet

In [ ]:
text = GENERATED_INPUT_PATH.read_text(encoding="utf-8")

print("\n" + "=" * 80)
print("First 120 Lines of Generated Input")
print("=" * 80)
lines = text.splitlines()
for i, line in enumerate(lines[:120], start=1):
    print(f"{i:04d}: {line}")

### Check generated geometry blocks

In [ ]:
n_polygonal_prisms = text.count("polygonal_prism")
n_region_blocks = text.count("region{")
n_contact_blocks = text.count("schottky{")

print("polygonal_prism count:", n_polygonal_prisms)
print("region{ count:", n_region_blocks)
print("schottky{ count:", n_contact_blocks)

In [ ]:
text = GENERATED_INPUT_PATH.read_text(encoding="utf-8")

checks = {
    "correct run quantum": "!WHEN $quantum            quantum{}" in text,
    "polygonal prisms": text.count("polygonal_prism"),
    "Body contact region": "contact{ name = Body }" in text,
    "remove_surface_charge region": "contact{ name = remove_surface_charge }" in text,
    "zero_fermi_QW region": "contact{ name = zero_fermi_QW }" in text,
    "top-level quantum blocks": text.count("\nquantum{"),
}

checks

## Run or select completed output

Use the generated input for an optional local run, or validate one explicitly selected transferred run.

### Internal paths and validation

Simulation staging and output remain outside the repository. Required outputs are conditional on the quantum-analysis control.


In [ ]:
RUN_TAG = "geometry_check_no_padding"

SIMULATION_OUTPUT_ROOT = Path(
    os.environ.get(
        "NEXTNANO_OUTPUT_ROOT",
        REPO_ROOT.parent
        / "qpu-local-outputs"
        / "refactoring"
        / "runs",
    )
).expanduser().resolve()
SIMULATION_STAGING_ROOT = SIMULATION_OUTPUT_ROOT / "_staging"

if (
    SIMULATION_OUTPUT_ROOT == REPO_ROOT
    or REPO_ROOT in SIMULATION_OUTPUT_ROOT.parents
):
    raise ValueError("SIMULATION_OUTPUT_ROOT must be outside the repository.")

BASE_REQUIRED_OUTPUTS = (
    "Structure/materials.vtr",
    "potential.vtr",
    "bandedges.vtr",
    "density_hole.vtr",
    "iteration_quantum_poisson.dat",
    "integrated_density_hole.dat",
    "total_charges.txt",
)
QUANTUM_REQUIRED_OUTPUTS = (
    "Quantum/c-Ge_QW/HH/density.vtr",
    "Quantum/c-Ge_QW/HH/probability_shift_k00000_0001.vtr",
    "Quantum/c-Ge_QW/HH/occupation.dat",
    "Quantum/c-Ge_QW/HH/energy_spectrum_k00000.dat",
)
REQUIRED_OUTPUTS = (
    BASE_REQUIRED_OUTPUTS
    + (QUANTUM_REQUIRED_OUTPUTS if ANALYSE_QUANTUM_OUTPUTS else ())
)


### Run or select

This single cell performs the complete local-versus-transferred choice and leaves one validated `RUN_DIRECTORY` for every following section.

In [ ]:
if RUN_SIMULATION:
    if RUN_DIRECTORY is not None:
        raise ValueError("Set RUN_DIRECTORY=None when RUN_SIMULATION=True.")

    if not GENERATED_INPUT_PATH.is_file():
        raise FileNotFoundError(
            f"Generated nextnano input file not found: {GENERATED_INPUT_PATH}"
        )

    if ANALYSE_QUANTUM_OUTPUTS:
        generated_input_text = GENERATED_INPUT_PATH.read_text(encoding="utf-8")
        quantum_solver_switches = {}
        for raw_line in generated_input_text.splitlines():
            uncommented_line = raw_line.split("#", 1)[0].strip()
            solver_switch_match = re.fullmatch(
                r"\$(quantum_poisson|quantum)\s*=\s*([01])",
                uncommented_line,
            )
            if solver_switch_match is not None:
                quantum_solver_switches[solver_switch_match.group(1)] = int(
                    solver_switch_match.group(2)
                )

        if not any(
            quantum_solver_switches.get(name, 0)
            for name in ("quantum", "quantum_poisson")
        ):
            raise RuntimeError(
                "Quantum-output analysis is enabled, but the generated input "
                "does not enable either $quantum or $quantum_poisson. "
                "The template/generated input must enable either the one-shot "
                "quantum solver ($quantum = 1) or the quantum–Poisson solver "
                "($quantum_poisson = 1) to produce the required quantum outputs. "
                "Solver settings are not changed automatically."
            )

    executed_input = run_input_file(
        GENERATED_INPUT_PATH,
        output_root=SIMULATION_OUTPUT_ROOT,
        tag=RUN_TAG,
        add_timestamp=True,
        show_log=True,
        convergenceCheck=False,
        staging_root=SIMULATION_STAGING_ROOT,
        keep_staged_input=True,
    )

    RUN_DIRECTORY = get_output_directory(executed_input)
else:
    if RUN_DIRECTORY is None:
        raise ValueError(
            "Set RUN_DIRECTORY to an explicit completed run when RUN_SIMULATION=False."
        )

RUN_DIRECTORY = validate_run_directory(
    RUN_DIRECTORY,
    bias=BIAS,
    require_complete=True,
    required_outputs=REQUIRED_OUTPUTS,
)
BIAS_DIRECTORY = get_bias_dir(RUN_DIRECTORY, BIAS)
STRUCTURE_DIRECTORY = RUN_DIRECTORY / "Structure"

print("Selected run directory:", RUN_DIRECTORY)
print("Selected bias directory:", BIAS_DIRECTORY)
print("Structure directory:", STRUCTURE_DIRECTORY)
print("Local execution enabled:", RUN_SIMULATION)
print("Configured local simulation output root:", SIMULATION_OUTPUT_ROOT)
print("Validated run directory:", RUN_DIRECTORY)

## Structure and diagnostics

The earlier PHIDL and process-stack views describe intended geometry. These cells inspect the structure and diagnostics from the explicitly validated solver run.

### Actual simulated-structure summary and mapping

The resolved files and lookup tables connect solver indices to material and contact names without hard-coding numerical indices.

In [ ]:
STRUCTURE_FILE_QUANTITIES = (
    "materials",
    "contacts",
    "regions_all",
)
STRUCTURE_FILES = {
    quantity: resolve_structure_file(
        RUN_DIRECTORY,
        quantity,
        preferred_extensions=("vtr",),
    )
    for quantity in STRUCTURE_FILE_QUANTITIES
}
STRUCTURE_FILE_SUMMARY = pd.DataFrame(
    [
        {"quantity": quantity, "filename": path.name, "path": path}
        for quantity, path in STRUCTURE_FILES.items()
    ]
)

MATERIAL_INDEX_PATH = resolve_structure_file(
    RUN_DIRECTORY,
    "material_indices",
    preferred_extensions=("txt",),
)
CONTACT_INDEX_PATH = resolve_structure_file(
    RUN_DIRECTORY,
    "contact_indices",
    preferred_extensions=("txt",),
)
MATERIAL_INDEX_TABLE = read_index_table(MATERIAL_INDEX_PATH)
CONTACT_INDEX_TABLE = read_index_table(CONTACT_INDEX_PATH)

display(STRUCTURE_FILE_SUMMARY)
display(MATERIAL_INDEX_TABLE)
display(CONTACT_INDEX_TABLE)

### Representative solver-structure views

The horizontal region plane uses the generated `xy_QD` section, while the vertical material plane uses the generated `xz_QD` section to verify the layer stack. The generated input does not define a compatible 1D structure section, so this workflow does not invent a structure line-cut coordinate.

In [ ]:
QUANTITY = "regions_all_2d_xy_QD"

HORIZONTAL_STRUCTURE_FIGURE = plot_structure_plane(
    RUN_DIRECTORY,
    quantity=QUANTITY,
    interactive=False,
)
display(HORIZONTAL_STRUCTURE_FIGURE)

In [ ]:
QUANTITY = "materials_2d_xz_QD"

VERTICAL_STRUCTURE_FIGURE = plot_structure_plane(
    RUN_DIRECTORY,
    quantity=QUANTITY,
    interactive=False,
)
display(VERTICAL_STRUCTURE_FIGURE)

### Quantum–Poisson convergence

Summarise the final residuals for the explicitly selected bias and plot the iteration history without displaying the complete raw table.

In [ ]:
CONVERGENCE_SUMMARY = convergence_summary(BIAS_DIRECTORY)
display(CONVERGENCE_SUMMARY)

CONVERGENCE_FIGURE = plot_convergence(
    BIAS_DIRECTORY,
    interactive=False,
)
display(CONVERGENCE_FIGURE)

### Integrated hole density

Read the run-level integrated density and show the final numerical values by solver region. Physical material labels require a compatible 1D structure DAT cut; this generated input defines only 2D structure sections, so the notebook does not guess that mapping.

In [ ]:
INTEGRATED_HOLE_DENSITY = read_integrated_density_hole(RUN_DIRECTORY)
INTEGRATED_REGION_COLUMNS = integrated_density_region_columns(
    INTEGRATED_HOLE_DENSITY
)

display(INTEGRATED_HOLE_DENSITY.loc[:, INTEGRATED_REGION_COLUMNS].tail(1))

INTEGRATED_HOLE_DENSITY_FIGURE = plot_integrated_density_hole(
    RUN_DIRECTORY,
    region_columns=INTEGRATED_REGION_COLUMNS,
    interactive=False,
    label_with_materials=False,
)
display(INTEGRATED_HOLE_DENSITY_FIGURE)

### Total charge

Read and display the selected bias's charge summary, then plot the reported charge quantities without saving the figure.

In [ ]:
TOTAL_CHARGES = read_total_charges(BIAS_DIRECTORY)
display(TOTAL_CHARGES)

TOTAL_CHARGES_FIGURE = plot_total_charges(
    BIAS_DIRECTORY,
    interactive=False,
)
display(TOTAL_CHARGES_FIGURE)

## Classical outputs

Default planes and line cuts use the corrected quantum-well depth and device centre line. Secondary bands, electron density, and full 3D views remain explicit opt-in diagnostics.

### Analysis defaults

These constants are the canonical spatial-cut provenance used by every plotting cell and by figure export.

In [ ]:
XY_PLANE_Z_NM = -4.0
XZ_PLANE_Y_NM = 0.0
X_LINE_Y_NM = 0.0
X_LINE_Z_NM = -4.0
Z_LINE_X_NM = 1150.0
Z_LINE_Y_NM = 0.0

SHOW_OPTIONAL_CLASSICAL_3D = False
SHOW_SECONDARY_BANDS = False
SHOW_ELECTRON_DENSITY = False

SECONDARY_BAND_VARIABLES = ("LH", "SO")

DENSITY_HOLE_VTR = resolve_bias_output_file(
    RUN_DIRECTORY,
    "density_hole",
    bias=BIAS,
    preferred_extensions=("vtr",),
)
POTENTIAL_VTR = resolve_bias_output_file(
    RUN_DIRECTORY,
    "potential",
    bias=BIAS,
    preferred_extensions=("vtr",),
)
BANDEDGES_VTR = resolve_bias_output_file(
    RUN_DIRECTORY,
    "bandedges",
    bias=BIAS,
    preferred_extensions=("vtr",),
)

CLASSICAL_OUTPUT_SUMMARY = pd.DataFrame(
    [
        {
            "quantity": quantity,
            "path": path,
            "variables": list_variables(path),
        }
        for quantity, path in (
            ("density_hole", DENSITY_HOLE_VTR),
            ("potential", POTENTIAL_VTR),
            ("bandedges", BANDEDGES_VTR),
        )
    ]
)
display(CLASSICAL_OUTPUT_SUMMARY)

### Classical hole density — xy plane

Show the lateral hole-density distribution at the configured quantum-well depth.

In [ ]:
QUANTITY = "density_hole"
VARIABLE = "Hole_density"
SLICE_AXIS = "z"
SLICE_COORDINATE_NM = XY_PLANE_Z_NM

HOLE_DENSITY_PLANE_DATA = load_vtr_plane(
    DENSITY_HOLE_VTR,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
)

HOLE_DENSITY_PLANE_FIGURE = plot_bias_volume_slice(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
    title=(
        "Classical hole density at "
        f"z = {HOLE_DENSITY_PLANE_DATA['slice_coordinate']:g} nm"
    ),
    interactive=False,
    log10=False,
)
display(HOLE_DENSITY_PLANE_FIGURE)

#### Classical hole density — x-axis line cut

In [ ]:
QUANTITY = "density_hole"
VARIABLE = "Hole_density"
LINE_AXIS = "x"
FIXED_COORDINATES_NM = {
    "y": X_LINE_Y_NM,
    "z": X_LINE_Z_NM,
}

HOLE_DENSITY_LINE_DATA = load_vtr_linecut(
    DENSITY_HOLE_VTR,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
)

HOLE_DENSITY_LINE_FIGURE = plot_bias_volume_linecut(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
    interactive=False,
)
HOLE_DENSITY_LINE_FIGURE.axes[0].set_title(
    f"Classical hole density along {LINE_AXIS} at "
    f"y = {HOLE_DENSITY_LINE_DATA['chosen_coords']['y']:g} nm, "
    f"z = {HOLE_DENSITY_LINE_DATA['chosen_coords']['z']:g} nm"
)
display(HOLE_DENSITY_LINE_FIGURE)

### Electrostatic potential — xy plane

Use the same quantum-well plane as the hole-density view.

In [ ]:
QUANTITY = "potential"
VARIABLE = "Potential"
SLICE_AXIS = "z"
SLICE_COORDINATE_NM = XY_PLANE_Z_NM

POTENTIAL_PLANE_DATA = load_vtr_plane(
    POTENTIAL_VTR,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
)

POTENTIAL_PLANE_FIGURE = plot_bias_volume_slice(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
    title=(
        "Electrostatic potential at "
        f"z = {POTENTIAL_PLANE_DATA['slice_coordinate']:g} nm"
    ),
    interactive=False,
    log10=False,
)
display(POTENTIAL_PLANE_FIGURE)

#### Electrostatic potential — xz plane

In [ ]:
QUANTITY = "potential"
VARIABLE = "Potential"
SLICE_AXIS = "y"
SLICE_COORDINATE_NM = XZ_PLANE_Y_NM

POTENTIAL_XZ_PLANE_DATA = load_vtr_plane(
    POTENTIAL_VTR,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
)
POTENTIAL_XZ_PLANE_FIGURE = plot_bias_volume_slice(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
    title=(
        "Electrostatic potential at "
        f"y = {POTENTIAL_XZ_PLANE_DATA['slice_coordinate']:g} nm"
    ),
    interactive=False,
    log10=False,
)
display(POTENTIAL_XZ_PLANE_FIGURE)

#### Electrostatic potential — x-axis line cut

In [ ]:
QUANTITY = "potential"
VARIABLE = "Potential"
LINE_AXIS = "x"
FIXED_COORDINATES_NM = {
    "y": X_LINE_Y_NM,
    "z": X_LINE_Z_NM,
}

POTENTIAL_LINE_DATA = load_vtr_linecut(
    POTENTIAL_VTR,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
)

POTENTIAL_LINE_FIGURE = plot_bias_volume_linecut(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
    interactive=False,
)
POTENTIAL_LINE_FIGURE.axes[0].set_title(
    f"Electrostatic potential along {LINE_AXIS} at "
    f"y = {POTENTIAL_LINE_DATA['chosen_coords']['y']:g} nm, "
    f"z = {POTENTIAL_LINE_DATA['chosen_coords']['z']:g} nm"
)
display(POTENTIAL_LINE_FIGURE)

### HH valence-band edge — xy plane

The HH band edge is the default valence-band result. LH and SO remain optional.

In [ ]:
QUANTITY = "bandedges"
VARIABLE = "HH"
SLICE_AXIS = "z"
SLICE_COORDINATE_NM = XY_PLANE_Z_NM

HH_BAND_PLANE_DATA = load_vtr_plane(
    BANDEDGES_VTR,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
)

HH_BAND_PLANE_FIGURE = plot_bias_volume_slice(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
    title=(
        "HH band edge at "
        f"z = {HH_BAND_PLANE_DATA['slice_coordinate']:g} nm"
    ),
    interactive=False,
    log10=False,
)
display(HH_BAND_PLANE_FIGURE)

#### HH valence-band edge — xz plane

In [ ]:
QUANTITY = "bandedges"
VARIABLE = "HH"
SLICE_AXIS = "y"
SLICE_COORDINATE_NM = XZ_PLANE_Y_NM

HH_BAND_XZ_PLANE_DATA = load_vtr_plane(
    BANDEDGES_VTR,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
)
HH_BAND_XZ_PLANE_FIGURE = plot_bias_volume_slice(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
    title=(
        "HH band edge at "
        f"y = {HH_BAND_XZ_PLANE_DATA['slice_coordinate']:g} nm"
    ),
    interactive=False,
    log10=False,
)
display(HH_BAND_XZ_PLANE_FIGURE)

#### HH valence-band edge — x-axis line cut

In [ ]:
QUANTITY = "bandedges"
VARIABLE = "HH"
LINE_AXIS = "x"
FIXED_COORDINATES_NM = {
    "y": X_LINE_Y_NM,
    "z": X_LINE_Z_NM,
}

HH_BAND_LINE_DATA = load_vtr_linecut(
    BANDEDGES_VTR,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
)

HH_BAND_LINE_FIGURE = plot_bias_volume_linecut(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
    interactive=False,
)
HH_BAND_LINE_FIGURE.axes[0].set_title(
    f"HH band edge along {LINE_AXIS} at "
    f"y = {HH_BAND_LINE_DATA['chosen_coords']['y']:g} nm, "
    f"z = {HH_BAND_LINE_DATA['chosen_coords']['z']:g} nm"
)
display(HH_BAND_LINE_FIGURE)

#### HH valence-band edge — z-axis line cut

In [ ]:
QUANTITY = "bandedges"
VARIABLE = "HH"
LINE_AXIS = "z"
FIXED_COORDINATES_NM = {
    "x": Z_LINE_X_NM,
    "y": Z_LINE_Y_NM,
}

HH_BAND_Z_LINE_DATA = load_vtr_linecut(
    BANDEDGES_VTR,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
)
HH_BAND_Z_LINE_FIGURE = plot_bias_volume_linecut(
    RUN_DIRECTORY,
    QUANTITY,
    bias=BIAS,
    variable=VARIABLE,
    axis=LINE_AXIS,
    fixed_coords=FIXED_COORDINATES_NM,
    interactive=False,
)
HH_BAND_Z_LINE_FIGURE.axes[0].set_title(
    f"HH band edge along {LINE_AXIS} at "
    f"x = {HH_BAND_Z_LINE_DATA['chosen_coords']['x']:g} nm, "
    f"y = {HH_BAND_Z_LINE_DATA['chosen_coords']['y']:g} nm"
)
display(HH_BAND_Z_LINE_FIGURE)

In [ ]:
SECONDARY_BAND_FIGURES = {}
if SHOW_SECONDARY_BANDS:
    QUANTITY = "bandedges"
    SLICE_AXIS = "z"
    SLICE_COORDINATE_NM = XY_PLANE_Z_NM
    LINE_AXIS = "x"
    FIXED_COORDINATES_NM = {
        "y": X_LINE_Y_NM,
        "z": X_LINE_Z_NM,
    }

    for band_variable in SECONDARY_BAND_VARIABLES:
        VARIABLE = band_variable
        band_plane_data = load_vtr_plane(
            BANDEDGES_VTR,
            variable=VARIABLE,
            slice_axis=SLICE_AXIS,
            slice_value=SLICE_COORDINATE_NM,
        )
        band_line_data = load_vtr_linecut(
            BANDEDGES_VTR,
            variable=VARIABLE,
            axis=LINE_AXIS,
            fixed_coords=FIXED_COORDINATES_NM,
        )

        band_plane_figure = plot_bias_volume_slice(
            RUN_DIRECTORY,
            QUANTITY,
            bias=BIAS,
            variable=VARIABLE,
            slice_axis=SLICE_AXIS,
            slice_value=SLICE_COORDINATE_NM,
            title=(
                f"{band_variable} band edge at "
                f"z = {band_plane_data['slice_coordinate']:g} nm"
            ),
            interactive=False,
            log10=False,
        )
        band_line_figure = plot_bias_volume_linecut(
            RUN_DIRECTORY,
            QUANTITY,
            bias=BIAS,
            variable=VARIABLE,
            axis=LINE_AXIS,
            fixed_coords=FIXED_COORDINATES_NM,
            interactive=False,
        )
        band_line_figure.axes[0].set_title(
            f"{band_variable} band edge along {LINE_AXIS} at "
            f"y = {band_line_data['chosen_coords']['y']:g} nm, "
            f"z = {band_line_data['chosen_coords']['z']:g} nm"
        )
        SECONDARY_BAND_FIGURES[band_variable] = {
            "plane": band_plane_figure,
            "line": band_line_figure,
        }
        display(band_plane_figure)
        display(band_line_figure)

### Optional classical views

Full 3D volumes are disabled by default because they are exploratory and comparatively expensive. Electron density is also disabled because `density_electron.vtr` is not a required output.

In [ ]:
CLASSICAL_3D_FIGURES = {}
if SHOW_OPTIONAL_CLASSICAL_3D:
    for figure_name, quantity, variable, title, z_range in (
        (
            "hole_density",
            "density_hole",
            "Hole_density",
            "Classical hole density near the quantum well",
            (-30.0, 170.0),
        ),
        (
            "potential",
            "potential",
            "Potential",
            "Electrostatic potential near the quantum well",
            (-30.0, 10.0),
        ),
        (
            "hh_band_edge",
            "bandedges",
            "HH",
            "HH band edge near the quantum well",
            (-30.0, 10.0),
        ),
    ):
        CLASSICAL_3D_FIGURES[figure_name] = plot_bias_volume_3d(
            RUN_DIRECTORY,
            quantity,
            bias=BIAS,
            variable=variable,
            title=title,
            log10=False,
            coord_ranges={"z": z_range},
            max_points=150_000,
            mode="volume",
            opacity=0.18,
            surface_count=10,
        )

In [ ]:
if SHOW_ELECTRON_DENSITY:
    QUANTITY = "density_electron"
    VARIABLE = "Electron_density"
    SLICE_AXIS = "z"
    SLICE_COORDINATE_NM = XY_PLANE_Z_NM
    LINE_AXIS = "x"
    FIXED_COORDINATES_NM = {
        "y": X_LINE_Y_NM,
        "z": X_LINE_Z_NM,
    }

    DENSITY_ELECTRON_VTR = resolve_bias_output_file(
        RUN_DIRECTORY,
        QUANTITY,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )
    electron_variables = list_variables(DENSITY_ELECTRON_VTR)
    if VARIABLE not in electron_variables:
        raise ValueError(
            f"Expected {VARIABLE!r} in {DENSITY_ELECTRON_VTR}; "
            f"available variables: {electron_variables}"
        )

    electron_plane_data = load_vtr_plane(
        DENSITY_ELECTRON_VTR,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
    )
    electron_line_data = load_vtr_linecut(
        DENSITY_ELECTRON_VTR,
        variable=VARIABLE,
        axis=LINE_AXIS,
        fixed_coords=FIXED_COORDINATES_NM,
    )

    ELECTRON_DENSITY_PLANE_FIGURE = plot_bias_volume_slice(
        RUN_DIRECTORY,
        QUANTITY,
        bias=BIAS,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
        title=(
            "Electron density at "
            f"z = {electron_plane_data['slice_coordinate']:g} nm"
        ),
        interactive=False,
        log10=False,
    )
    ELECTRON_DENSITY_LINE_FIGURE = plot_bias_volume_linecut(
        RUN_DIRECTORY,
        QUANTITY,
        bias=BIAS,
        variable=VARIABLE,
        axis=LINE_AXIS,
        fixed_coords=FIXED_COORDINATES_NM,
        interactive=False,
    )
    ELECTRON_DENSITY_LINE_FIGURE.axes[0].set_title(
        f"Electron density along {LINE_AXIS} at "
        f"y = {electron_line_data['chosen_coords']['y']:g} nm, "
        f"z = {electron_line_data['chosen_coords']['z']:g} nm"
    )
    display(ELECTRON_DENSITY_PLANE_FIGURE)
    display(ELECTRON_DENSITY_LINE_FIGURE)

    if SHOW_OPTIONAL_CLASSICAL_3D:
        ELECTRON_DENSITY_3D_FIGURE = plot_bias_volume_3d(
            RUN_DIRECTORY,
            QUANTITY,
            bias=BIAS,
            variable=VARIABLE,
            title="Electron density near the quantum well",
            log10=False,
            coord_ranges={"z": (-30.0, 170.0)},
            max_points=150_000,
            mode="volume",
            opacity=0.18,
            surface_count=10,
        )

### HH band-edge minimum

Report the minimum HH band edge in the same configured QW plane as a concise numerical result.

In [ ]:
QUANTITY = "bandedges"
VARIABLE = "HH"
SLICE_AXIS = "z"
SLICE_COORDINATE_NM = XY_PLANE_Z_NM

HH_QW_PLANE_MINIMUM = find_vtr_plane_extrema(
    BANDEDGES_VTR,
    variable=VARIABLE,
    slice_axis=SLICE_AXIS,
    slice_value=SLICE_COORDINATE_NM,
    kind="min",
    n_extrema=1,
)
display(
    HH_QW_PLANE_MINIMUM[
        ["x_nm", "y_nm", "z_nm", "value", "value_label"]
    ]
)

## Quantum outputs

`ANALYSE_QUANTUM_OUTPUTS` controls whether the selected run must contain the configured HH results. When enabled, missing results fail clearly; when disabled, the classical-only workflow continues and this analysis is skipped. Plane and line views reuse the classical coordinates configured above.


In [ ]:
QUANTUM_REGION = "c-Ge_QW"
QUANTUM_BAND = "HH"
QUANTUM_KPOINT = "k00000"
QUANTUM_STATE = 1
SHOW_OPTIONAL_QUANTUM_3D = False
EXPECTED_PROBABILITY_PEAKS = 2
PROBABILITY_PEAK_MIN_SEPARATION_NM = 50.0

if ANALYSE_QUANTUM_OUTPUTS:
    QUANTUM_DENSITY_VTR = resolve_quantum_output_file(
        RUN_DIRECTORY,
        "density",
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )
    QUANTUM_PROBABILITY_SHIFT_VTR = resolve_quantum_probability_state_file(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        shifted=True,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )

    QUANTUM_OUTPUT_SUMMARY = pd.DataFrame(
        [
            {
                "result": f"{QUANTUM_BAND} quantum hole density",
                "file": str(QUANTUM_DENSITY_VTR.relative_to(BIAS_DIRECTORY)),
                "variables": ", ".join(list_variables(QUANTUM_DENSITY_VTR)),
            },
            {
                "result": f"Shifted state {QUANTUM_STATE} probability density",
                "file": str(QUANTUM_PROBABILITY_SHIFT_VTR.relative_to(BIAS_DIRECTORY)),
                "variables": ", ".join(list_variables(QUANTUM_PROBABILITY_SHIFT_VTR)),
            },
        ]
    )
    display(QUANTUM_OUTPUT_SUMMARY)
else:
    display("Quantum analysis is disabled (ANALYSE_QUANTUM_OUTPUTS=False).")


### Quantum-calculated HH density — xy plane

Show the quantum HH density at the same corrected QW depth as the classical plane.


In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTITY = "density"
    VARIABLE = "Density"
    SLICE_AXIS = "z"
    SLICE_COORDINATE_NM = XY_PLANE_Z_NM

    QUANTUM_DENSITY_PLANE_FIGURE = plot_quantum_density_volume_slice(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
        title=(
            f"Quantum-calculated {QUANTUM_BAND} hole density at "
            f"z = {SLICE_COORDINATE_NM:g} nm"
        ),
        interactive=False,
        log10=False,
    )
    display(QUANTUM_DENSITY_PLANE_FIGURE)

#### Quantum-calculated HH density — x-axis line cut

In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTITY = "density"
    VARIABLE = "Density"
    LINE_AXIS = "x"
    FIXED_COORDINATES_NM = {
        "y": X_LINE_Y_NM,
        "z": X_LINE_Z_NM,
    }

    QUANTUM_DENSITY_LINE_FIGURE = plot_quantum_density_volume_linecut(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        variable=VARIABLE,
        axis=LINE_AXIS,
        fixed_coords=FIXED_COORDINATES_NM,
        title=(
            f"Quantum-calculated {QUANTUM_BAND} hole density along {LINE_AXIS} "
            f"at y = {FIXED_COORDINATES_NM['y']:g} nm, "
            f"z = {FIXED_COORDINATES_NM['z']:g} nm"
        ),
        interactive=False,
    )
    display(QUANTUM_DENSITY_LINE_FIGURE)


### Shifted state probability density — xy plane

The configured shifted state is shown in the same QW plane and along the matching classical line. The peak summary requests two separated maxima because the configured layout contains two dots; it uses the established 50 nm lateral separation from the repository's existing double-dot analysis and labels the results as probability-density peaks, not inferred dot centres.


In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTITY = "probability_shift"
    VARIABLE = f"Psi^2_{QUANTUM_STATE}"
    SLICE_AXIS = "z"
    SLICE_COORDINATE_NM = XY_PLANE_Z_NM

    QUANTUM_PROBABILITY_PLANE_FIGURE = plot_quantum_probability_volume_slice(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        shifted=True,
        bias=BIAS,
        variable=VARIABLE,
        slice_axis=SLICE_AXIS,
        slice_value=SLICE_COORDINATE_NM,
        title=(
            f"Shifted probability density for state {QUANTUM_STATE} at "
            f"z = {SLICE_COORDINATE_NM:g} nm"
        ),
        interactive=False,
        log10=False,
    )
    display(QUANTUM_PROBABILITY_PLANE_FIGURE)

#### Shifted state probability density — x-axis line cut

In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTITY = "probability_shift"
    VARIABLE = f"Psi^2_{QUANTUM_STATE}"
    LINE_AXIS = "x"
    FIXED_COORDINATES_NM = {
        "y": X_LINE_Y_NM,
        "z": X_LINE_Z_NM,
    }

    QUANTUM_PROBABILITY_LINE_FIGURE = plot_quantum_probability_volume_linecut(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        shifted=True,
        bias=BIAS,
        variable=VARIABLE,
        axis=LINE_AXIS,
        fixed_coords=FIXED_COORDINATES_NM,
        title=(
            f"Shifted probability density for state {QUANTUM_STATE} along "
            f"{LINE_AXIS} at y = {FIXED_COORDINATES_NM['y']:g} nm, "
            f"z = {FIXED_COORDINATES_NM['z']:g} nm"
        ),
        interactive=False,
    )
    display(QUANTUM_PROBABILITY_LINE_FIGURE)


In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    if SHOW_OPTIONAL_QUANTUM_3D:
        DENSITY_QUANTITY = "density"
        DENSITY_VARIABLE = "Density"
        PROBABILITY_QUANTITY = "probability_shift"
        PROBABILITY_VARIABLE = f"Psi^2_{QUANTUM_STATE}"

        QUANTUM_DENSITY_VOLUME_FIGURE = plot_quantum_density_volume_3d(
            RUN_DIRECTORY,
            region=QUANTUM_REGION,
            band=QUANTUM_BAND,
            bias=BIAS,
            variable=DENSITY_VARIABLE,
            title=f"Quantum-calculated {QUANTUM_BAND} hole density near the QW",
            log10=False,
            coord_ranges={"z": (-30.0, 10.0)},
            max_points=150_000,
            mode="volume",
            opacity=0.18,
            surface_count=10,
        )
        QUANTUM_PROBABILITY_VOLUME_FIGURE = plot_quantum_probability_volume_3d(
            RUN_DIRECTORY,
            state=QUANTUM_STATE,
            region=QUANTUM_REGION,
            band=QUANTUM_BAND,
            kpoint=QUANTUM_KPOINT,
            shifted=True,
            bias=BIAS,
            variable=PROBABILITY_VARIABLE,
            title=f"Shifted probability density for state {QUANTUM_STATE} near the QW",
            log10=False,
            coord_ranges={"z": (-30.0, 10.0)},
            max_points=150_000,
            mode="volume",
            opacity=0.18,
            surface_count=10,
        )


In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTITY = "probability_shift"
    VARIABLE = f"Psi^2_{QUANTUM_STATE}"

    QUANTUM_PROBABILITY_PEAKS = find_probability_peaks(
        QUANTUM_PROBABILITY_SHIFT_VTR,
        variable=VARIABLE,
        n_peaks=EXPECTED_PROBABILITY_PEAKS,
        min_lateral_separation_nm=PROBABILITY_PEAK_MIN_SEPARATION_NM,
    )
    QUANTUM_PROBABILITY_PEAKS_SUMMARY = QUANTUM_PROBABILITY_PEAKS.loc[
        :, ["peak", "x_nm", "y_nm", "z_nm", "probability"]
    ]
    display(QUANTUM_PROBABILITY_PEAKS_SUMMARY)


### State occupation

The table keeps the state coordinate and occupation value used by the public plotter, limited to the first ten rows.


In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTUM_OCCUPATION = read_quantum_occupation(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
    )
    QUANTUM_OCCUPATION_COLUMNS = list(
        dict.fromkeys(
            (
                QUANTUM_OCCUPATION.columns[0],
                QUANTUM_OCCUPATION.columns[-1],
            )
        )
    )
    QUANTUM_OCCUPATION_SUMMARY = QUANTUM_OCCUPATION.loc[
        :, QUANTUM_OCCUPATION_COLUMNS
    ].head(10)
    display(QUANTUM_OCCUPATION_SUMMARY)
    QUANTUM_OCCUPATION_FIGURE = plot_quantum_occupation(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        interactive=False,
    )
    display(QUANTUM_OCCUPATION_FIGURE)


### Energy spectrum

The configured k-point spectrum is summarized with the state coordinate and energy value used by the public plotter.


In [ ]:
if ANALYSE_QUANTUM_OUTPUTS:
    QUANTUM_ENERGY_SPECTRUM = read_quantum_energy_spectrum(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        bias=BIAS,
    )
    QUANTUM_ENERGY_COLUMNS = list(
        dict.fromkeys(
            (
                QUANTUM_ENERGY_SPECTRUM.columns[0],
                QUANTUM_ENERGY_SPECTRUM.columns[-1],
            )
        )
    )
    QUANTUM_ENERGY_SUMMARY = QUANTUM_ENERGY_SPECTRUM.loc[
        :, QUANTUM_ENERGY_COLUMNS
    ].head(10)
    display(QUANTUM_ENERGY_SUMMARY)
    QUANTUM_ENERGY_FIGURE = plot_quantum_energy_spectrum(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        bias=BIAS,
        interactive=False,
    )
    display(QUANTUM_ENERGY_FIGURE)


## Figure export

Collect the default static Matplotlib figures under stable filenames. Export remains disabled unless `EXPORT_FIGURES=True`; optional Plotly 3D and disabled diagnostic figures are deliberately excluded.

### Figure export settings

In [ ]:
# THESIS_FIGURE_ROOT may point directly to the thesis figure location
# on the analysis computer.
FIGURE_OUTPUT_ROOT = Path(
    os.environ.get(
        "THESIS_FIGURE_ROOT",
        REPO_ROOT.parent
        / "qpu-local-outputs"
        / "refactoring"
        / "figures",
    )
).expanduser().resolve()
FIGURE_DIRECTORY = (
    FIGURE_OUTPUT_ROOT
    / "notebook_03_lateral_double_quantum_dot"
)
FIGURE_FORMATS = ("pdf", "png")
FIGURE_PNG_DPI = 300

if FIGURE_OUTPUT_ROOT == REPO_ROOT or REPO_ROOT in FIGURE_OUTPUT_ROOT.parents:
    raise ValueError("FIGURE_OUTPUT_ROOT must be outside the repository.")

In [ ]:
THESIS_FIGURES = {
    "structure_xy_qw": HORIZONTAL_STRUCTURE_FIGURE,
    "structure_xz_qw": VERTICAL_STRUCTURE_FIGURE,
    "convergence": CONVERGENCE_FIGURE,
    "integrated_hole_density": INTEGRATED_HOLE_DENSITY_FIGURE,
    "total_charge": TOTAL_CHARGES_FIGURE,
    "classical_hole_density_xy_qw": HOLE_DENSITY_PLANE_FIGURE,
    "classical_hole_density_x_cut": HOLE_DENSITY_LINE_FIGURE,
    "electrostatic_potential_xy_qw": POTENTIAL_PLANE_FIGURE,
    "electrostatic_potential_xz_center": POTENTIAL_XZ_PLANE_FIGURE,
    "electrostatic_potential_x_cut": POTENTIAL_LINE_FIGURE,
    "hh_bandedge_xy_qw": HH_BAND_PLANE_FIGURE,
    "hh_bandedge_xz_center": HH_BAND_XZ_PLANE_FIGURE,
    "hh_bandedge_x_cut": HH_BAND_LINE_FIGURE,
    "hh_bandedge_z_cut_x_1150": HH_BAND_Z_LINE_FIGURE,
}

if ANALYSE_QUANTUM_OUTPUTS:
    THESIS_FIGURES.update(
        {
            "quantum_hh_density_xy_qw": QUANTUM_DENSITY_PLANE_FIGURE,
            "quantum_hh_density_x_cut": QUANTUM_DENSITY_LINE_FIGURE,
            f"hh_probability_state_{QUANTUM_STATE:04d}_xy_qw": (
                QUANTUM_PROBABILITY_PLANE_FIGURE
            ),
            f"hh_probability_state_{QUANTUM_STATE:04d}_x_cut": (
                QUANTUM_PROBABILITY_LINE_FIGURE
            ),
            "hh_occupation": QUANTUM_OCCUPATION_FIGURE,
            f"hh_energy_spectrum_{QUANTUM_KPOINT}": QUANTUM_ENERGY_FIGURE,
        }
    )

### Controlled export and provenance

When enabled, export each mapped Matplotlib figure as PDF and high-resolution PNG, then record concise analysis provenance and relative figure filenames in a deterministic JSON manifest.


In [ ]:
if EXPORT_FIGURES:
    unsupported_figure_formats = sorted(
        set(FIGURE_FORMATS) - {"pdf", "png"}
    )
    if unsupported_figure_formats:
        raise ValueError(
            "Unsupported figure format(s): "
            f"{unsupported_figure_formats}. Use only 'pdf' and 'png'."
        )

    FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    exported_figure_records = []
    exported_figure_files = {}

    for figure_stem, figure in THESIS_FIGURES.items():
        is_matplotlib_figure = (
            type(figure).__module__ == "matplotlib.figure"
            and type(figure).__name__ == "Figure"
        )
        if not is_matplotlib_figure:
            raise TypeError(
                f"THESIS_FIGURES[{figure_stem!r}] must be a Matplotlib "
                f"Figure, not {type(figure).__module__}."
                f"{type(figure).__name__}."
            )

        figure_filenames = []
        for figure_format in FIGURE_FORMATS:
            figure_filename = f"{figure_stem}.{figure_format}"
            figure_path = FIGURE_DIRECTORY / figure_filename
            if figure_format == "png":
                figure.savefig(
                    figure_path,
                    bbox_inches="tight",
                    dpi=FIGURE_PNG_DPI,
                )
            else:
                figure.savefig(figure_path, bbox_inches="tight")

            figure_filenames.append(figure_filename)
            exported_figure_records.append(
                {
                    "figure": figure_stem,
                    "relative_path": figure_filename,
                }
            )
        exported_figure_files[figure_stem] = figure_filenames

    FIGURE_MANIFEST = {
        "notebook_identifier": (
            "03_generate_nextnano_input_from_phidl_layout"
        ),
        "run_directory": str(RUN_DIRECTORY),
        "bias": BIAS,
        "generated_input_path": (
            str(GENERATED_INPUT_PATH) if RUN_SIMULATION else None
        ),
        "xy_plane_z_nm": XY_PLANE_Z_NM,
        "xz_plane_y_nm": XZ_PLANE_Y_NM,
        "x_line_fixed_coordinates_nm": {
            "y": X_LINE_Y_NM,
            "z": X_LINE_Z_NM,
        },
        "z_line_fixed_coordinates_nm": {
            "x": Z_LINE_X_NM,
            "y": Z_LINE_Y_NM,
        },
        "analyse_quantum_outputs": ANALYSE_QUANTUM_OUTPUTS,
        "quantum": (
            {
                "region": QUANTUM_REGION,
                "band": QUANTUM_BAND,
                "kpoint": QUANTUM_KPOINT,
                "state": QUANTUM_STATE,
            }
            if ANALYSE_QUANTUM_OUTPUTS
            else None
        ),
        "figure_files": exported_figure_files,
        "formats": list(FIGURE_FORMATS),
        "png_dpi": FIGURE_PNG_DPI,
    }
    FIGURE_MANIFEST_PATH = FIGURE_DIRECTORY / "figure_manifest.json"
    FIGURE_MANIFEST_PATH.write_text(
        json.dumps(FIGURE_MANIFEST, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    EXPORTED_FIGURES = pd.DataFrame(exported_figure_records)
    display(EXPORTED_FIGURES)

## Summary

Summarise the selected run, scientific cut configuration, quantum-analysis mode, and controlled figure-export destination without dumping raw data or configuration.


In [ ]:
NOTEBOOK_FINAL_SUMMARY = pd.DataFrame(
    [
        {"setting": "run_directory", "value": str(RUN_DIRECTORY)},
        {"setting": "bias", "value": BIAS},
        {"setting": "xy_plane_z_nm", "value": XY_PLANE_Z_NM},
        {"setting": "xz_plane_y_nm", "value": XZ_PLANE_Y_NM},
        {
            "setting": "x_line_fixed_coordinates_nm",
            "value": json.dumps(
                {"y": X_LINE_Y_NM, "z": X_LINE_Z_NM},
                sort_keys=True,
            ),
        },
        {
            "setting": "z_line_fixed_coordinates_nm",
            "value": json.dumps(
                {"x": Z_LINE_X_NM, "y": Z_LINE_Y_NM},
                sort_keys=True,
            ),
        },
        {
            "setting": "quantum_analysis_enabled",
            "value": ANALYSE_QUANTUM_OUTPUTS,
        },
        {
            "setting": "figures_available_for_export",
            "value": len(THESIS_FIGURES),
        },
        {
            "setting": "figure_directory",
            "value": str(FIGURE_DIRECTORY),
        },
        {"setting": "export_enabled", "value": EXPORT_FIGURES},
    ]
)
display(NOTEBOOK_FINAL_SUMMARY)